# Notebook 02: Residual Stream Analysis (Level 2a)

## Overview
Analyzes how **frequency band structure evolves through the residual stream**
across transformer layers. While NB01 examined static embeddings, this notebook
tracks representational geometry and discriminability at each layer.

## Key Questions
- At which layer does band discriminability peak?
- Does the separation ratio increase monotonically through layers?
- Can a linear "frequency direction" be tracked across layers?
- How do trajectories compare across model sizes?

## Hypothesis Domain: R2 (Residual Stream)
- **H-R2.1**: Probe accuracy increases through layers (Jonckheere-Terpstra, per model)
- **H-R2.2**: Separation ratio increases through layers (trend test, per model)
- **H-R2.3**: Frequency direction R² > permutation null (per model)
- **H-R2.4**: Convergence layer differs by band (Kruskal-Wallis, per model)
- **H-R2.5**: Low-frequency inputs converge later than high-frequency (Mann-Whitney, per model)

## Notebook Structure
1. Setup & Data Loading
2. Residual Stream Geometry Trajectory
3. Within-Band Variance Trajectory
4. Separation Ratio Trajectory
5. Frequency Direction Tracking
6. Linear Probe Trajectory
7. MLP Probe Comparison (key layers)
8. RSA Trajectory
9. CKA Across Layers
10. Cross-Model Comparison
11. Draw Stability

## Data Sources
- Pre-extracted activations: `resid_post_predpos` from NPZ files
- Shape per file: (N_examples, n_layers, d_model)

## 1. Setup & Data Loading

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from utils.constants import (
    MODELS,
    BANDS,
    DRAWS,
    FREQUENCY_RANK,
    MODEL_INFO,
    BAND_COLORS,
    BAND_NAMES,
    MODEL_COLORS,
    MODEL_CAPACITY,
    MODEL_D_MODEL,
    ACTIVATIONS_DIR,
    ANALYSIS_DIR,
    VIZ_DIR,
    RANDOM_SEED,
    N_PERMUTATIONS,
    CV_FOLDS,
    get_domain_dirs,
)
from utils.data_loading import (
    load_extracted_activations,
    save_analysis,
    build_representational_df,
)
from utils.geometry import (
    compute_band_centroids,
    compute_centroid_distances,
    compute_within_band_spread,
    compute_separation_ratio,
    compute_participation_ratio,
    compute_isotropy,
    linear_cka,
    compute_rsa_similarity,
)
from utils.probing import train_probe, train_probe_trajectory, train_mlp_probe
from utils.plotting import (
    setup_plotting,
    save_figure,
    plot_probe_trajectory,
    plot_metric_heatmap,
)

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

setup_plotting()
ANALYSIS_DIR, VIZ_DIR = get_domain_dirs("residual_stream", "base")
from functools import partial as _partial

save_analysis = _partial(save_analysis, analysis_dir=ANALYSIS_DIR)
save_figure = _partial(save_figure, viz_dir=VIZ_DIR)

print(f"Models: {MODELS}")
print(f"Bands: {BANDS}")
print(f"Draws: {DRAWS}")

Models: ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b']
Bands: ['low', 'medium', 'high', 'very_high', 'control']
Draws: ['draw_1', 'draw_2', 'draw_3']


In [2]:
# Load residual stream activations at prediction position
all_resid = {}  # model -> draw -> {band: (N, n_layers, d_model)}

for model in MODELS:
    all_resid[model] = {}
    for draw in DRAWS:
        all_resid[model][draw] = {}
        for band in BANDS:
            try:
                data = load_extracted_activations(model, band, draw)
                all_resid[model][draw][band] = data[
                    "resid_post_predpos"
                ]  # (N, n_layers, d_model)
            except FileNotFoundError:
                pass

# Report shapes
for model in MODELS:
    sample = next(iter(next(iter(all_resid[model].values())).values()), None)
    if sample is not None:
        print(f"{model}: resid shape = {sample.shape} (N, n_layers, d_model)")

pythia-70m: resid shape = (225, 6, 512) (N, n_layers, d_model)
pythia-160m: resid shape = (225, 12, 768) (N, n_layers, d_model)
pythia-410m: resid shape = (225, 24, 1024) (N, n_layers, d_model)
pythia-1b: resid shape = (225, 16, 2048) (N, n_layers, d_model)
pythia-1.4b: resid shape = (225, 24, 2048) (N, n_layers, d_model)


## 2. Residual Stream Geometry Trajectory

Norms, participation ratio, and isotropy at each layer.

In [3]:
geom_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        for band in BANDS:
            resid = all_resid.get(model, {}).get(draw, {}).get(band)
            if resid is None:
                continue

            for layer in range(n_layers):
                X = resid[:, layer, :]  # (N, d_model)
                norms = np.linalg.norm(X, axis=1)
                pr = compute_participation_ratio(X)
                iso = compute_isotropy(X)

                geom_records.append(
                    {
                        "model": model,
                        "draw": draw,
                        "band": band,
                        "layer": layer,
                        "mean_norm": float(norms.mean()),
                        "std_norm": float(norms.std()),
                        "participation_ratio": pr,
                        "isotropy": iso,
                    }
                )

df_geom = pd.DataFrame(geom_records)
save_analysis(df_geom, "02_resid_geometry_trajectory.csv")
print(f"Geometry records: {len(df_geom)}")

Geometry records: 1230


In [4]:
# Plot norm trajectory per model, colored by band
for model in MODELS:
    model_data = df_geom[df_geom["model"] == model]
    if len(model_data) == 0:
        continue

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    metrics = ["mean_norm", "participation_ratio", "isotropy"]
    titles = ["Mean L2 Norm", "Participation Ratio", "Isotropy"]

    for ax, metric, title in zip(axes, metrics, titles):
        for band in BANDS:
            bd = model_data[
                (model_data["band"] == band) & (model_data["draw"] == "draw_1")
            ]
            if len(bd) == 0:
                continue
            bd = bd.sort_values("layer")
            ax.plot(
                bd["layer"],
                bd[metric],
                color=BAND_COLORS.get(band, "gray"),
                label=BAND_NAMES.get(band, band),
                marker="o",
                markersize=3,
            )
        ax.set_xlabel("Layer")
        ax.set_title(f"{title}: {model}")

    axes[0].legend(fontsize=7)
    fig.tight_layout()
    save_figure(fig, f"viz_02_01_geometry_trajectory_{model}.png")

## 3. Within-Band Variance Trajectory

Trace of covariance matrix per band per layer. Does variance evolve differently
for low vs high frequency?

In [5]:
var_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        for band in BANDS:
            resid = all_resid.get(model, {}).get(draw, {}).get(band)
            if resid is None:
                continue

            for layer in range(n_layers):
                X = resid[:, layer, :]
                cov_trace = np.trace(np.cov(X.T)) if X.shape[0] > 1 else 0
                total_var = np.var(X, axis=0).sum()

                var_records.append(
                    {
                        "model": model,
                        "draw": draw,
                        "band": band,
                        "layer": layer,
                        "cov_trace": float(cov_trace),
                        "total_variance": float(total_var),
                    }
                )

df_var = pd.DataFrame(var_records)
save_analysis(df_var, "02_variance_trajectory.csv")
print(f"Variance records: {len(df_var)}")

Variance records: 1230


In [6]:
# Plot variance trajectory per model
for model in MODELS:
    model_data = df_var[(df_var["model"] == model) & (df_var["draw"] == "draw_1")]
    if len(model_data) == 0:
        continue

    fig, ax = plt.subplots(figsize=(10, 5))
    for band in BANDS:
        bd = model_data[model_data["band"] == band].sort_values("layer")
        if len(bd) == 0:
            continue
        ax.plot(
            bd["layer"],
            bd["total_variance"],
            color=BAND_COLORS.get(band, "gray"),
            label=BAND_NAMES.get(band, band),
            marker="o",
            markersize=3,
        )
    ax.set_xlabel("Layer")
    ax.set_ylabel("Total Variance")
    ax.set_title(f"Within-Band Variance Trajectory: {model}")
    ax.legend(fontsize=8)
    save_figure(fig, f"viz_02_02_variance_trajectory_{model}.png")

## 4. Separation Ratio Trajectory

Band separation at each layer. Does it increase through the network?

In [7]:
sep_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        for layer in range(n_layers):
            embs, labels = [], []
            for band in BANDS:
                resid = all_resid.get(model, {}).get(draw, {}).get(band)
                if resid is not None:
                    embs.append(resid[:, layer, :])
                    labels.extend([band] * resid.shape[0])

            if len(embs) < 2:
                continue

            X = np.vstack(embs)
            y = np.array(labels)
            centroids = compute_band_centroids(X, y)
            distances = compute_centroid_distances(centroids)
            spreads = compute_within_band_spread(X, y)
            sep = compute_separation_ratio(distances, spreads)

            sep_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": layer,
                    "separation_ratio": sep,
                }
            )

df_sep = pd.DataFrame(sep_records)
save_analysis(df_sep, "02_separation_trajectory.csv")
print(f"Separation records: {len(df_sep)}")

Separation records: 246


In [8]:
# Plot separation ratio trajectory
fig, ax = plt.subplots(figsize=(12, 6))
for model in MODELS:
    model_data = df_sep[
        (df_sep["model"] == model) & (df_sep["draw"] == "draw_1")
    ].sort_values("layer")
    if len(model_data) == 0:
        continue
    ax.plot(
        model_data["layer"],
        model_data["separation_ratio"],
        color=MODEL_COLORS.get(model, "gray"),
        label=model,
        marker="o",
        markersize=4,
    )

ax.set_xlabel("Layer")
ax.set_ylabel("Separation Ratio")
ax.set_title("Band Separation Ratio Across Layers")
ax.legend()
save_figure(fig, "viz_02_03_separation_trajectory.png")

## 5. Frequency Direction Tracking

Ridge regression predicting frequency rank from residual stream, per layer.
Measures how much the model builds a "frequency direction" in activation space.

In [9]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold

freq_dir_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        for layer in range(n_layers):
            embs, freq_ranks = [], []
            for band in BANDS:
                resid = all_resid.get(model, {}).get(draw, {}).get(band)
                # Skip bands without a numeric frequency rank (e.g. 'control' -> None)
                if resid is None or FREQUENCY_RANK.get(band) is None:
                    continue
                embs.append(resid[:, layer, :])
                freq_ranks.extend([FREQUENCY_RANK[band]] * resid.shape[0])

            if len(embs) < 2:
                continue

            X = np.vstack(embs)
            y = np.array(freq_ranks, dtype=float)

            # Cross-validated Ridge regression R²
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)
            ridge = Ridge(alpha=1.0)
            cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
            scores = cross_val_score(ridge, X_scaled, y, cv=cv, scoring="r2")

            freq_dir_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": layer,
                    "r2_mean": float(scores.mean()),
                    "r2_std": float(scores.std()),
                }
            )

df_freq_dir = pd.DataFrame(freq_dir_records)
save_analysis(df_freq_dir, "02_freq_direction_r2.csv")
print(f"Frequency direction records: {len(df_freq_dir)}")

Frequency direction records: 246


In [10]:
# Plot frequency direction R² trajectory
fig, ax = plt.subplots(figsize=(12, 6))
for model in MODELS:
    model_data = df_freq_dir[
        (df_freq_dir["model"] == model) & (df_freq_dir["draw"] == "draw_1")
    ].sort_values("layer")
    if len(model_data) == 0:
        continue
    ax.plot(
        model_data["layer"],
        model_data["r2_mean"],
        color=MODEL_COLORS.get(model, "gray"),
        label=model,
        marker="o",
        markersize=4,
    )
    ax.fill_between(
        model_data["layer"],
        model_data["r2_mean"] - model_data["r2_std"],
        model_data["r2_mean"] + model_data["r2_std"],
        color=MODEL_COLORS.get(model, "gray"),
        alpha=0.15,
    )

ax.set_xlabel("Layer")
ax.set_ylabel("Ridge R² (Frequency Rank)")
ax.set_title("Frequency Direction Strength Across Layers")
ax.legend()
ax.set_ylim(bottom=-0.1)
save_figure(fig, "viz_02_04_freq_direction_trajectory.png")

## 6. Linear Probe Trajectory

5-fold CV accuracy at each layer. When does band discriminability peak?

In [11]:
probe_trajectory_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    print(f"\n{model} ({n_layers} layers):")

    for draw in DRAWS:
        # Build per-layer activation dict
        layer_activations = {}
        labels_list = []

        for layer in range(n_layers):
            embs = []
            if layer == 0:
                labels_list = []
            for band in BANDS:
                resid = all_resid.get(model, {}).get(draw, {}).get(band)
                if resid is None:
                    continue
                embs.append(resid[:, layer, :])
                if layer == 0:
                    labels_list.extend([band] * resid.shape[0])

            if len(embs) > 0:
                layer_activations[layer] = np.vstack(embs)

        labels = np.array(labels_list)

        if not layer_activations:
            continue

        # Train probes at all layers
        trajectory = train_probe_trajectory(layer_activations, labels)

        for entry in trajectory:
            probe_trajectory_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": entry["layer"],
                    "accuracy": entry["accuracy"],
                    "std": entry["std"],
                    "band": "all",  # All bands combined
                }
            )

        peak = max(trajectory, key=lambda x: x["accuracy"])
        print(
            f"  {draw}: peak accuracy = {peak['accuracy']:.3f} at layer {peak['layer']}"
        )

df_probe_traj = pd.DataFrame(probe_trajectory_records)
save_analysis(df_probe_traj, "02_probe_trajectory.csv")
print(f"\nProbe trajectory records: {len(df_probe_traj)}")


pythia-70m (6 layers):


  draw_1: peak accuracy = 0.611 at layer 3


  draw_2: peak accuracy = 0.581 at layer 2


  draw_3: peak accuracy = 0.588 at layer 4

pythia-160m (12 layers):


  draw_1: peak accuracy = 0.663 at layer 7


...

  draw_2: peak accuracy = 0.692 at layer 15


  draw_3: peak accuracy = 0.682 at layer 15

pythia-1.4b (24 layers):


  draw_1: peak accuracy = 0.726 at layer 23


  draw_2: peak accuracy = 0.735 at layer 23


  draw_3: peak accuracy = 0.746 at layer 23

Probe trajectory records: 246


In [12]:
# Plot probe trajectories
fig, ax = plt.subplots(figsize=(12, 6))
for model in MODELS:
    model_data = df_probe_traj[
        (df_probe_traj["model"] == model) & (df_probe_traj["draw"] == "draw_1")
    ].sort_values("layer")
    if len(model_data) == 0:
        continue
    ax.plot(
        model_data["layer"],
        model_data["accuracy"],
        color=MODEL_COLORS.get(model, "gray"),
        label=model,
        marker="o",
        markersize=4,
    )
    if "std" in model_data.columns:
        ax.fill_between(
            model_data["layer"],
            model_data["accuracy"] - model_data["std"],
            model_data["accuracy"] + model_data["std"],
            color=MODEL_COLORS.get(model, "gray"),
            alpha=0.15,
        )

ax.axhline(y=1.0 / len(BANDS), color="gray", linestyle="--", alpha=0.5, label="Chance")
ax.set_xlabel("Layer")
ax.set_ylabel("Probe Accuracy (5-fold CV)")
ax.set_title("Linear Probe Trajectory: Band Classification")
ax.legend()
ax.set_ylim(bottom=0)
save_figure(fig, "viz_02_05_probe_trajectory.png")

## 7. MLP Probe Comparison at Key Layers

Is band information linearly or non-linearly encoded?

In [13]:
mlp_probe_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    # Test at first, middle, and last layers
    key_layers = [0, n_layers // 4, n_layers // 2, 3 * n_layers // 4, n_layers - 1]
    key_layers = sorted(set(key_layers))

    for draw in ["draw_1"]:  # Single draw for efficiency
        for layer in key_layers:
            embs, labels = [], []
            for band in BANDS:
                resid = all_resid.get(model, {}).get(draw, {}).get(band)
                if resid is None:
                    continue
                embs.append(resid[:, layer, :])
                labels.extend([band] * resid.shape[0])

            if len(embs) < 2:
                continue

            X = np.vstack(embs)
            y = np.array(labels)

            linear_result = train_probe(X, y)
            mlp_result = train_mlp_probe(X, y)

            mlp_probe_records.append(
                {
                    "model": model,
                    "layer": layer,
                    "linear_accuracy": linear_result["accuracy"],
                    "mlp_accuracy": mlp_result["accuracy"],
                    "linearity_gap": mlp_result["accuracy"] - linear_result["accuracy"],
                }
            )

df_mlp_probe = pd.DataFrame(mlp_probe_records)
print("Linear vs MLP probe at key layers:")
print(df_mlp_probe.to_string(index=False))

Linear vs MLP probe at key layers:
      model  layer  linear_accuracy  mlp_accuracy  linearity_gap
 pythia-70m      0         0.552000      0.547556      -0.004444
 pythia-70m      1         0.557333      0.559111       0.001778
 pythia-70m      3         0.610667      0.566222      -0.044444
 pythia-70m      4         0.603556      0.557333      -0.046222
 pythia-70m      5         0.580444      0.551111      -0.029333
pythia-160m      0         0.569778      0.540444      -0.029333
pythia-160m      3         0.601778      0.560000      -0.041778
pythia-160m      6         0.652444      0.627556      -0.024889
pythia-160m      9         0.621333      0.616889      -0.004444
pythia-160m     11         0.623111      0.599111      -0.024000
pythia-410m      0         0.559111      0.526222      -0.032889
pythia-410m      6         0.566222      0.544000      -0.022222
pythia-410m     12         0.590222      0.560889      -0.029333
pythia-410m     18         0.654222      0.598222      

## 8. RSA Trajectory

Representational Similarity Analysis at each layer: Spearman correlation
between pairwise distance matrices of activations.

In [14]:
from scipy.spatial.distance import pdist, squareform

rsa_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in ["draw_1"]:
        for layer in range(n_layers):
            # Compute band centroids at this layer
            # Only include bands with a numeric frequency rank (skip 'control' -> None)
            band_centroids = {}
            for band in BANDS:
                resid = all_resid.get(model, {}).get(draw, {}).get(band)
                if resid is not None and FREQUENCY_RANK.get(band) is not None:
                    band_centroids[band] = resid[:, layer, :].mean(axis=0)

            if len(band_centroids) < 3:
                continue

            bands_ordered = sorted(
                band_centroids.keys(), key=lambda b: FREQUENCY_RANK[b]
            )

            # RDM from activation centroids
            centroid_matrix = np.array([band_centroids[b] for b in bands_ordered])
            rdm_act = squareform(pdist(centroid_matrix, metric="euclidean"))

            # RDM from frequency ranks
            ranks = np.array([FREQUENCY_RANK[b] for b in bands_ordered], dtype=float)
            rdm_freq = squareform(pdist(ranks.reshape(-1, 1), metric="euclidean"))

            rsa_rho, rsa_p = compute_rsa_similarity(rdm_act, rdm_freq)

            rsa_records.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": layer,
                    "rsa_rho": rsa_rho,
                    "rsa_p": rsa_p,
                }
            )

df_rsa = pd.DataFrame(rsa_records)
save_analysis(df_rsa, "02_rsa_trajectory.csv")
print(f"RSA records: {len(df_rsa)}")

RSA records: 82


In [15]:
# Plot RSA trajectory
fig, ax = plt.subplots(figsize=(12, 6))
for model in MODELS:
    model_data = df_rsa[df_rsa["model"] == model].sort_values("layer")
    if len(model_data) == 0:
        continue
    ax.plot(
        model_data["layer"],
        model_data["rsa_rho"],
        color=MODEL_COLORS.get(model, "gray"),
        label=model,
        marker="o",
        markersize=4,
    )

ax.set_xlabel("Layer")
ax.set_ylabel("RSA (Spearman rho)")
ax.set_title("RSA: Activation Distance vs Frequency Distance")
ax.legend()
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.3)
save_figure(fig, "viz_02_06_rsa_trajectory.png")

## 9. CKA Across Layers

Layer-to-layer CKA within the same model. Which layers have similar representations?

In [16]:
for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]

    # Combine all bands for draw_1
    embs = []
    for band in BANDS:
        resid = all_resid.get(model, {}).get("draw_1", {}).get(band)
        if resid is not None:
            embs.append(resid)

    if len(embs) == 0:
        continue

    all_resid_stack = np.concatenate(embs, axis=0)  # (N_total, n_layers, d_model)

    # Compute layer-to-layer CKA
    cka_matrix = np.zeros((n_layers, n_layers))
    for i in range(n_layers):
        for j in range(i, n_layers):
            cka_val = linear_cka(all_resid_stack[:, i, :], all_resid_stack[:, j, :])
            cka_matrix[i, j] = cka_val
            cka_matrix[j, i] = cka_val

    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(
        cka_matrix,
        annot=False,
        cmap="RdYlBu_r",
        square=True,
        linewidths=0,
        linecolor="none",
        vmin=0,
        vmax=1,
        ax=ax,
        xticklabels=range(n_layers),
        yticklabels=range(n_layers),
    )
    ax.set_xlabel("Layer")
    ax.set_ylabel("Layer")
    ax.set_title(f"Layer-to-Layer CKA: {model}")
    save_figure(fig, f"viz_02_07_layer_cka_{model}.png")

## 10. Cross-Model Comparison

Probe trajectories and separation trajectories overlaid across model sizes.

In [17]:
# Build master residual DataFrame
master_records = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        # Peak probe accuracy and layer
        probe_data = df_probe_traj[
            (df_probe_traj["model"] == model) & (df_probe_traj["draw"] == draw)
        ]
        if len(probe_data) > 0:
            peak_row = probe_data.loc[probe_data["accuracy"].idxmax()]
            peak_acc = peak_row["accuracy"]
            peak_layer = peak_row["layer"]
        else:
            peak_acc, peak_layer = np.nan, np.nan

        # Peak separation ratio
        sep_data = df_sep[(df_sep["model"] == model) & (df_sep["draw"] == draw)]
        if len(sep_data) > 0:
            peak_sep = sep_data["separation_ratio"].max()
            peak_sep_layer = sep_data.loc[
                sep_data["separation_ratio"].idxmax(), "layer"
            ]
        else:
            peak_sep, peak_sep_layer = np.nan, np.nan

        master_records.append(
            {
                "model": model,
                "draw": draw,
                "model_capacity": MODEL_CAPACITY[model],
                "n_layers": n_layers,
                "peak_probe_accuracy": peak_acc,
                "peak_probe_layer": peak_layer,
                "peak_probe_layer_frac": peak_layer / n_layers
                if not np.isnan(peak_layer)
                else np.nan,
                "peak_separation_ratio": peak_sep,
                "peak_separation_layer": peak_sep_layer,
            }
        )

df_master_resid = pd.DataFrame(master_records)
save_analysis(df_master_resid, "02_master_residual.csv")
print(
    df_master_resid.groupby("model")[
        ["peak_probe_accuracy", "peak_probe_layer_frac", "peak_separation_ratio"]
    ]
    .mean()
    .round(3)
)

             peak_probe_accuracy  peak_probe_layer_frac  peak_separation_ratio
model                                                                         
pythia-1.4b                0.736                  0.958                 19.645
pythia-160m                0.648                  0.556                 12.816
pythia-1b                  0.691                  0.938                 16.240
pythia-410m                0.688                  0.944                 13.712
pythia-70m                 0.593                  0.500                 12.058


## 11. Draw Stability

In [18]:
stability_records = []
for model in MODELS:
    model_data = df_master_resid[df_master_resid["model"] == model]
    for metric in [
        "peak_probe_accuracy",
        "peak_probe_layer_frac",
        "peak_separation_ratio",
    ]:
        vals = model_data[metric].dropna()
        if len(vals) >= 2:
            stability_records.append(
                {
                    "model": model,
                    "metric": metric,
                    "mean": float(vals.mean()),
                    "std": float(vals.std()),
                    "cv": float(vals.std() / vals.mean())
                    if vals.mean() != 0
                    else np.nan,
                }
            )

df_stab = pd.DataFrame(stability_records)
save_analysis(df_stab, "02_draw_stability.csv")
print("Draw stability:")
if len(df_stab) > 0:
    print(df_stab.pivot(index="model", columns="metric", values="cv").round(3))

Draw stability:
metric       peak_probe_accuracy  peak_probe_layer_frac  peak_separation_ratio
model                                                                         
pythia-1.4b                0.013                  0.000                  0.016
pythia-160m                0.029                  0.087                  0.030
pythia-1b                  0.012                  0.000                  0.012
pythia-410m                0.009                  0.025                  0.096
pythia-70m                 0.026                  0.333                  0.179


In [19]:
print("\n" + "=" * 70)
print("NOTEBOOK 02 COMPLETE")
print("=" * 70)
print(f"\nOutput CSVs in: {ANALYSIS_DIR}")
print(f"Figures in: {VIZ_DIR}")
for f in sorted(ANALYSIS_DIR.glob("02_*")):
    print(f"  {f.name}")
for f in sorted(VIZ_DIR.glob("viz_02_*")):
    print(f"  {f.name}")


NOTEBOOK 02 COMPLETE

Output CSVs in: LSC_circuit_analysis/03_Phase_Representational/outputs/residual_stream/base/analysis
Figures in: LSC_circuit_analysis/03_Phase_Representational/outputs/residual_stream/base/viz
  02_draw_stability.csv
  02_freq_direction_r2.csv
  02_master_residual.csv
  02_probe_trajectory.csv
  02_resid_geometry_trajectory.csv
  02_rsa_trajectory.csv
  02_separation_trajectory.csv
  02_variance_trajectory.csv
  viz_02_01_geometry_trajectory_pythia-1.4b.png
  viz_02_01_geometry_trajectory_pythia-160m.png
  viz_02_01_geometry_trajectory_pythia-1b.png
  viz_02_01_geometry_trajectory_pythia-410m.png
  viz_02_01_geometry_trajectory_pythia-70m.png
  viz_02_02_variance_trajectory_pythia-1.4b.png
  viz_02_02_variance_trajectory_pythia-160m.png
  viz_02_02_variance_trajectory_pythia-1b.png
  viz_02_02_variance_trajectory_pythia-410m.png
  viz_02_02_variance_trajectory_pythia-70m.png
  viz_02_03_separation_trajectory.png
  viz_02_04_freq_direction_trajectory.png
  viz_02_